# Journal-scale SLE validation on 128 real identities

This notebook consolidates the biometric and security evidence needed
for the journal version of Synchronized Locator Evolution (SLE).

It extends the preliminary experiment in four concrete ways:

1. 128 real CelebA identities are enrolled, with six images per identity.
2. Closed-set identification and verification metrics are reported.
3. Learned linkability attacks are evaluated on 15 independent held-out
   runs, rather than two runs.
4. Confidence intervals use cluster bootstrap over complete runs, so
   correlated epoch transitions are not treated as independent samples.

The notebook is intentionally self-contained. It streams CelebA,
extracts FaceNet/VGGFace2 embeddings on GPU, discards the floating-point
embedding matrix after binary-root construction, executes the SLE
semantics and fault tests, and then runs the CPU security benchmark.

Claim boundary: the experiments support empirical unlinkability under
the stated observable-token model. They are not a cryptographic proof,
and SLE does not protect a reconstructed biometric root, a reconstructed
epoch key, or public persistent bucket metadata.


In [ ]:
# CELL 1. Journal experiment controls
SEED = 20260721

# Real biometric experiment.
N_IDENTITIES = 128
IMAGES_PER_ID = 6
GALLERY_IMAGES_PER_ID = 2
EMBEDDING_BITS = 512
INDEX_BLOCK_BITS = 8
N_EPOCHS = 20

CELEBA_DATASET = 'flwrlabs/celeba'
CELEBA_SPLIT = 'train'
IMAGE_KEY = 'image'
IDENTITY_KEY = 'celeb_id'
MAX_STREAM_ROWS = 170_000

# Security experiment. Five complete runs are used only for training;
# fifteen different runs and their late epochs are held out.
SECURITY_N_VALUES = [32, 64, 128]
N_TRAIN_RUNS = 5
N_TEST_RUNS = 15
N_SECURITY_RUNS = N_TRAIN_RUNS + N_TEST_RUNS
TRAIN_RUNS = list(range(N_TRAIN_RUNS))
TEST_RUNS = list(range(N_TRAIN_RUNS, N_SECURITY_RUNS))
TRAIN_EPOCHS = list(range(0, 12))
TEST_EPOCHS = list(range(12, 20))
NEGATIVES_PER_POSITIVE = 4

EXTRA_TREES = 120
EXTRA_TREES_MAX_DEPTH = 14
CLUSTER_BOOTSTRAP_SAMPLES = 3000

BUCKET_LEAK_FRACTIONS = [0.0, 0.02, 0.05, 0.10, 0.25, 0.50, 1.0]
BUCKET_LEAK_TRIALS = 60

USE_SYNTHETIC_SMOKE = False
OUTPUT_DIR = '/content/sle_journal_validation_outputs'

print({
    'real_identities': N_IDENTITIES,
    'images_per_identity': IMAGES_PER_ID,
    'security_N': SECURITY_N_VALUES,
    'training_runs': len(TRAIN_RUNS),
    'held_out_runs': len(TEST_RUNS),
    'epochs': N_EPOCHS,
})


## Environment

`facenet-pytorch` is installed without dependencies. This avoids the
NumPy/PyTorch binary incompatibility caused by replacing Colab's core
scientific stack. CelebA is accessed in streaming mode; no Kaggle token
and no full local dataset reconstruction are required.


In [ ]:
# CELL 2. Safe dependencies and imports
import importlib.util
import subprocess
import sys


def install_if_missing(module_name, pip_args):
    if importlib.util.find_spec(module_name) is None:
        print('Installing:', ' '.join(pip_args))
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pip_args])
    else:
        print(module_name, 'is already available')


install_if_missing('facenet_pytorch', ['--no-deps', 'facenet-pytorch==2.6.0'])
install_if_missing('datasets', ['datasets'])

import copy
import hashlib
import hmac
import io
import json
import math
import shutil
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Image as HFImage
from datasets import load_dataset
from facenet_pytorch import InceptionResnetV1
from IPython.display import display
from PIL import Image as PILImage
from scipy.optimize import linear_sum_assignment
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from tqdm.auto import tqdm

OUTPUT = Path(OUTPUT_DIR)
OUTPUT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Device:', DEVICE)
print('PyTorch:', torch.__version__)
print('NumPy:', np.__version__)


## Two-pass CelebA streaming

The first pass requests only identity labels and counts how many train
images are available per person. A deterministic sample of 128 eligible
identities is then selected. The second pass retains compressed image
bytes only for those identities. This controls RAM usage while avoiding
selection based on facial appearance or recognition quality.


In [ ]:
# CELL 3. Select 128 identities and collect exactly six images each
def image_record_to_bytes(record):
    if isinstance(record, dict):
        if record.get('bytes') is not None:
            return bytes(record['bytes'])
        if record.get('path'):
            return Path(record['path']).read_bytes()
    if isinstance(record, PILImage.Image):
        buffer = io.BytesIO()
        record.convert('RGB').save(buffer, format='JPEG', quality=95)
        return buffer.getvalue()
    raise ValueError(f'Unsupported image record: {type(record)}')


def decode_rgb(image_bytes):
    with PILImage.open(io.BytesIO(image_bytes)) as image:
        return image.convert('RGB').copy()


def make_synthetic_faces():
    # This mode verifies notebook wiring only. Never report it as a
    # biometric result in the paper.
    rng = np.random.default_rng(SEED)
    images = {}
    for identity in range(N_IDENTITIES):
        base = rng.integers(35, 220, size=(1, 1, 3), dtype=np.uint8)
        samples = []
        for _ in range(IMAGES_PER_ID):
            noise = rng.normal(0, 16, size=(160, 160, 3))
            array = np.clip(base + noise, 0, 255).astype(np.uint8)
            samples.append(PILImage.fromarray(array, mode='RGB'))
        images[identity] = samples
    return images


def collect_real_faces():
    print('Pass 1/2: counting identity labels...')
    label_stream = load_dataset(
        CELEBA_DATASET,
        split=CELEBA_SPLIT,
        streaming=True,
    ).select_columns([IDENTITY_KEY])

    counts = Counter()
    inspected = 0
    for row in label_stream:
        counts[int(row[IDENTITY_KEY])] += 1
        inspected += 1
        if inspected % 25_000 == 0:
            eligible_now = sum(value >= IMAGES_PER_ID for value in counts.values())
            print(f'labels={inspected:,}, eligible identities={eligible_now:,}')
        if inspected >= MAX_STREAM_ROWS:
            break

    eligible = sorted(
        identity for identity, count in counts.items()
        if count >= IMAGES_PER_ID
    )
    if len(eligible) < N_IDENTITIES:
        raise RuntimeError(
            f'Only {len(eligible)} identities have at least {IMAGES_PER_ID} images.'
        )

    rng = np.random.default_rng(SEED)
    selected_ids = sorted(
        rng.choice(eligible, size=N_IDENTITIES, replace=False).astype(int).tolist()
    )
    selected_set = set(selected_ids)
    print('Selected identities:', selected_ids[:10], '...')

    print('Pass 2/2: collecting compressed images for selected identities...')
    image_stream = load_dataset(
        CELEBA_DATASET,
        split=CELEBA_SPLIT,
        streaming=True,
    ).select_columns([IMAGE_KEY, IDENTITY_KEY])
    image_stream = image_stream.cast_column(IMAGE_KEY, HFImage(decode=False))

    collected = {identity: [] for identity in selected_ids}
    complete = set()
    inspected = 0
    for row in image_stream:
        inspected += 1
        identity = int(row[IDENTITY_KEY])
        if identity in selected_set and len(collected[identity]) < IMAGES_PER_ID:
            collected[identity].append(image_record_to_bytes(row[IMAGE_KEY]))
            if len(collected[identity]) == IMAGES_PER_ID:
                complete.add(identity)

        if inspected % 25_000 == 0:
            print(
                f'images={inspected:,}, complete identities='
                f'{len(complete)}/{N_IDENTITIES}'
            )
        if len(complete) == N_IDENTITIES or inspected >= MAX_STREAM_ROWS:
            break

    if len(complete) != N_IDENTITIES:
        missing = sorted(selected_set - complete)
        raise RuntimeError(f'Incomplete selected identities: {missing[:10]}')

    images = {
        identity: [decode_rgb(value) for value in collected[identity]]
        for identity in selected_ids
    }
    del collected
    return images


images_by_identity = (
    make_synthetic_faces() if USE_SYNTHETIC_SMOKE else collect_real_faces()
)
selected_ids = sorted(images_by_identity)
print('Final identity count:', len(selected_ids))


In [ ]:
# CELL 4. Visual identity sanity check
preview_ids = selected_ids[:4]
fig, axes = plt.subplots(len(preview_ids), IMAGES_PER_ID, figsize=(12, 7))
for row, identity in enumerate(preview_ids):
    for column, image in enumerate(images_by_identity[identity]):
        axes[row, column].imshow(image)
        axes[row, column].axis('off')
        if column == 0:
            axes[row, column].set_title(f'Identity {identity}', loc='left', fontsize=9)
fig.suptitle('CelebA journal subset: repeated observations per identity', fontsize=13)
fig.tight_layout()
fig.savefig(OUTPUT / '01_identity_samples.png', dpi=220, bbox_inches='tight')
plt.show()


## FaceNet embeddings and binary biometric roots

Two images per identity form the enrollment gallery template. Four
different images are retained as probes. FaceNet/VGGFace2 produces a
normalized 512-dimensional embedding; the sign of each dimension forms
a 512-bit binary root. The complete floating-point matrix is deleted
after root construction.


In [ ]:
# CELL 5. GPU embedding extraction and irreversible experiment handoff
facenet_transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

flat_images = []
flat_labels = []
flat_roles = []
for identity in selected_ids:
    for position, image in enumerate(images_by_identity[identity]):
        flat_images.append(facenet_transform(image))
        flat_labels.append(identity)
        flat_roles.append('gallery' if position < GALLERY_IMAGES_PER_ID else 'probe')

labels_array = np.asarray(flat_labels, dtype=np.int64)
role_array = np.asarray(flat_roles)
image_tensor = torch.stack(flat_images)
loader = DataLoader(TensorDataset(image_tensor), batch_size=64, shuffle=False)

facenet = InceptionResnetV1(pretrained='vggface2').eval().to(DEVICE)
embedding_batches = []
with torch.inference_mode():
    for (batch,) in tqdm(loader, desc='FaceNet/VGGFace2 embeddings'):
        embedding = facenet(batch.to(DEVICE))
        embedding_batches.append(F.normalize(embedding, dim=1).cpu())
all_embeddings = torch.cat(embedding_batches).numpy().astype(np.float32)

gallery_roots = []
gallery_labels = []
probe_roots = []
probe_labels = []
for identity in selected_ids:
    gallery_mask = (labels_array == identity) & (role_array == 'gallery')
    probe_mask = (labels_array == identity) & (role_array == 'probe')
    mean_gallery = all_embeddings[gallery_mask].mean(axis=0)
    gallery_roots.append((mean_gallery >= 0).astype(np.uint8))
    gallery_labels.append(identity)
    identity_probes = (all_embeddings[probe_mask] >= 0).astype(np.uint8)
    probe_roots.extend(identity_probes)
    probe_labels.extend([identity] * len(identity_probes))

gallery_roots = np.asarray(gallery_roots, dtype=np.uint8)
gallery_labels = np.asarray(gallery_labels, dtype=np.int64)
probe_roots = np.asarray(probe_roots, dtype=np.uint8)
probe_labels = np.asarray(probe_labels, dtype=np.int64)

assert gallery_roots.shape == (N_IDENTITIES, EMBEDDING_BITS)
assert probe_roots.shape == (
    N_IDENTITIES * (IMAGES_PER_ID - GALLERY_IMAGES_PER_ID),
    EMBEDDING_BITS,
)

np.savez_compressed(
    OUTPUT / 'celeba_128_binary_roots.npz',
    gallery_roots=gallery_roots,
    gallery_labels=gallery_labels,
    probe_roots=probe_roots,
    probe_labels=probe_labels,
)
print('Saved root checkpoint:', OUTPUT / 'celeba_128_binary_roots.npz')

del all_embeddings, embedding_batches, image_tensor, flat_images, facenet, images_by_identity
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Floating-point embedding matrix deleted; binary roots retained.')


## Biometric utility and verification

Identification is evaluated by gallery/probe Rank-1 and Rank-5.
Verification is evaluated from genuine and impostor Hamming-similarity
distributions using EER and TAR at FAR 1e-2 and 1e-3. Rank-1 uncertainty
is bootstrapped by identity, preserving the four correlated probes that
belong to each person.


In [ ]:
# CELL 6. Journal biometric metrics and identity-cluster bootstrap
def hamming_distances(query_root, enrolled_roots):
    return np.count_nonzero(enrolled_roots != query_root[None, :], axis=1)


distance_matrix = np.count_nonzero(
    probe_roots[:, None, :] != gallery_roots[None, :, :],
    axis=2,
)
ranking = np.argsort(distance_matrix, axis=1, kind='stable')
baseline_predictions = gallery_labels[ranking[:, 0]]
rank1_correct = baseline_predictions == probe_labels
rank1 = float(rank1_correct.mean())
rank5 = float(np.mean([
    probe_labels[index] in gallery_labels[ranking[index, :5]]
    for index in range(len(probe_labels))
]))

label_to_gallery = {int(label): index for index, label in enumerate(gallery_labels)}
genuine_columns = np.asarray([label_to_gallery[int(label)] for label in probe_labels])
similarity_matrix = 1.0 - distance_matrix.astype(np.float32) / EMBEDDING_BITS
genuine_scores = similarity_matrix[np.arange(len(probe_labels)), genuine_columns]
impostor_mask = np.ones_like(similarity_matrix, dtype=bool)
impostor_mask[np.arange(len(probe_labels)), genuine_columns] = False
impostor_scores = similarity_matrix[impostor_mask]

verification_labels = np.concatenate([
    np.ones(len(genuine_scores), dtype=np.uint8),
    np.zeros(len(impostor_scores), dtype=np.uint8),
])
verification_scores = np.concatenate([genuine_scores, impostor_scores])
fpr, tpr, thresholds = roc_curve(verification_labels, verification_scores)
fnr = 1.0 - tpr
eer_index = int(np.argmin(np.abs(fpr - fnr)))
eer = float((fpr[eer_index] + fnr[eer_index]) / 2.0)
eer_threshold = float(thresholds[eer_index])

def tar_at_far(target_far):
    valid = tpr[fpr <= target_far]
    return float(valid.max()) if len(valid) else 0.0


rng = np.random.default_rng(SEED + 600)
identity_values = np.asarray(sorted(set(probe_labels.tolist())))
per_identity_accuracy = np.asarray([
    rank1_correct[probe_labels == identity].mean()
    for identity in identity_values
])
bootstrap_draws = rng.choice(
    per_identity_accuracy,
    size=(5000, len(per_identity_accuracy)),
    replace=True,
).mean(axis=1)
rank1_ci_low, rank1_ci_high = np.quantile(bootstrap_draws, [0.025, 0.975])

biometric_metrics_df = pd.DataFrame([{
    'identities': N_IDENTITIES,
    'gallery_images_per_identity': GALLERY_IMAGES_PER_ID,
    'probe_images': len(probe_roots),
    'rank1': rank1,
    'rank1_ci_low': float(rank1_ci_low),
    'rank1_ci_high': float(rank1_ci_high),
    'rank5': rank5,
    'verification_eer': eer,
    'eer_threshold': eer_threshold,
    'tar_at_far_1e-2': tar_at_far(1e-2),
    'tar_at_far_1e-3': tar_at_far(1e-3),
}])
display(biometric_metrics_df)

# Repeated identity subsets show how gallery size affects Rank-1.
subset_rows = []
for subset_n in SECURITY_N_VALUES:
    for repeat in range(20):
        subset_rng = np.random.default_rng(SEED + subset_n * 100 + repeat)
        subset_labels = np.sort(
            subset_rng.choice(gallery_labels, size=subset_n, replace=False)
        )
        gallery_mask = np.isin(gallery_labels, subset_labels)
        probe_mask = np.isin(probe_labels, subset_labels)
        subset_distances = np.count_nonzero(
            probe_roots[probe_mask, None, :] != gallery_roots[gallery_mask][None, :, :],
            axis=2,
        )
        subset_predictions = gallery_labels[gallery_mask][np.argmin(subset_distances, axis=1)]
        subset_rows.append({
            'N': subset_n,
            'repeat': repeat,
            'rank1': float(np.mean(subset_predictions == probe_labels[probe_mask])),
        })
utility_subset_df = pd.DataFrame(subset_rows)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3))
axes[0].hist(impostor_scores, bins=45, density=True, alpha=0.65, label='Impostor')
axes[0].hist(genuine_scores, bins=30, density=True, alpha=0.65, label='Genuine')
axes[0].axvline(eer_threshold, color='black', linestyle='--', linewidth=1)
axes[0].set_xlabel('Binary-root similarity')
axes[0].set_ylabel('Density')
axes[0].set_title('(a) Verification score distributions')
axes[0].legend(frameon=False)
axes[0].grid(alpha=0.2)

subset_summary = utility_subset_df.groupby('N')['rank1'].agg(['mean', 'std']).reset_index()
axes[1].errorbar(
    subset_summary['N'], subset_summary['mean'], yerr=subset_summary['std'],
    marker='o', capsize=4, color='#3B6EA8',
)
axes[1].set_xlabel('Enrolled identities, N')
axes[1].set_ylabel('Rank-1 identification accuracy')
axes[1].set_ylim(0, 1.02)
axes[1].set_title('(b) Utility across identity subsets')
axes[1].grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT / '02_biometric_utility.png', dpi=240, bbox_inches='tight')
plt.show()


## SLE semantics, hidden relation validation, and rollback

The complete 128-identity gallery is indexed by 8-bit root blocks. The
database copy and every index copy of a locator are evolved by the same
keyed permutation. Two multiset fingerprints validate the hidden
`(locator, block, value)` relation before atomic commit. Six controlled
corruptions test rejection and rollback.


In [ ]:
# CELL 9. State representation: protected database plus locator index
@dataclass
class SLEState:
    epoch: int
    records: list
    index: dict


def root_blocks(root, block_bits=INDEX_BLOCK_BITS):
    if len(root) % block_bits != 0:
        raise ValueError('Root length must be divisible by block size')
    blocks = []
    for start in range(0, len(root), block_bits):
        value = 0
        for offset, bit in enumerate(root[start:start + block_bits]):
            value |= int(bit) << offset
        blocks.append(value)
    return blocks


def unique_initial_locators(count, seed):
    rng = np.random.default_rng(seed)
    values = []
    seen = set()
    while len(values) < count:
        value = int(rng.integers(0, 2**63, dtype=np.uint64))
        if value not in seen:
            seen.add(value)
            values.append(value)
    return values


def build_state(roots, labels):
    locators = unique_initial_locators(len(labels), SEED + 101)
    records = []
    index = defaultdict(list)
    for locator, root, label in zip(locators, roots, labels):
        records.append({
            'locator': int(locator),
            'label': int(label),
            'root': np.asarray(root, dtype=np.uint8).copy(),
        })
        for block, value in enumerate(root_blocks(root)):
            index[(block, value)].append(int(locator))
    return SLEState(epoch=0, records=records, index=dict(index))


def indexed_predictions(state, probes):
    by_locator = {record['locator']: record for record in state.records}
    predictions = []
    candidate_sizes = []
    true_candidate_flags = []

    for probe_index, probe in enumerate(probes):
        candidates = set()
        for block, value in enumerate(root_blocks(probe)):
            candidates.update(state.index.get((block, value), []))

        if not candidates:
            candidates = set(by_locator)

        candidate_records = [by_locator[value] for value in sorted(candidates)]
        candidate_roots = np.stack([record['root'] for record in candidate_records])
        candidate_labels = np.asarray([record['label'] for record in candidate_records])
        distances = hamming_distances(probe, candidate_roots)
        best = np.flatnonzero(distances == distances.min())
        winner = best[np.argmin(candidate_labels[best])]
        predictions.append(int(candidate_labels[winner]))
        candidate_sizes.append(len(candidate_records))
        true_candidate_flags.append(int(probe_labels[probe_index] in candidate_labels))

    return (
        np.asarray(predictions, dtype=np.int64),
        np.asarray(candidate_sizes),
        np.asarray(true_candidate_flags),
    )


state0 = build_state(gallery_roots, gallery_labels)
indexed_before, candidate_sizes, candidate_recall = indexed_predictions(
    state0, probe_roots
)
indexed_rank1 = float(np.mean(indexed_before == probe_labels))

print(f'Indexed Rank-1: {indexed_rank1:.4f}')
print(f'Mean candidate set: {candidate_sizes.mean():.2f} / {N_IDENTITIES}')
print(f'True identity candidate recall: {candidate_recall.mean():.4f}')
print('Indexed/full-scan decision agreement:', float(np.mean(indexed_before == baseline_predictions)))


In [ ]:
# CELL 10. Key derivation, 64-bit PRP, and relation fingerprints
MASK32 = (1 << 32) - 1
MASK64 = (1 << 64) - 1
FINGERPRINT_PRIME = (1 << 127) - 1


def distributed_epoch_key(epoch, master_seed=SEED):
    # Five deterministic contributions make the notebook reproducible.
    # In the MPC protocol each contribution is private to one party.
    contributions = []
    for party in range(5):
        message = f'SLE|seed={master_seed}|epoch={epoch}|party={party}'.encode()
        contributions.append(hashlib.sha256(message).digest())
    return hashlib.sha256(b''.join(contributions)).digest()


def feistel_prp64(value, key, epoch, rounds=8):
    left = (int(value) >> 32) & MASK32
    right = int(value) & MASK32
    for round_no in range(rounds):
        payload = (
            int(epoch).to_bytes(8, 'big')
            + int(round_no).to_bytes(2, 'big')
            + int(right).to_bytes(4, 'big')
        )
        function_value = int.from_bytes(
            hmac.new(key, payload, hashlib.sha256).digest()[:4], 'big'
        )
        left, right = right, (left ^ function_value) & MASK32
    return ((left << 32) | right) & MASK64


def relation_tuples(state):
    expected = []
    for record in state.records:
        for block, value in enumerate(root_blocks(record['root'])):
            expected.append((int(record['locator']), block, value))

    actual = []
    for (block, value), locator_list in state.index.items():
        for locator in locator_list:
            actual.append((int(locator), int(block), int(value)))
    return expected, actual


def encode_relation_tuple(item):
    payload = '|'.join(map(str, item)).encode()
    return int.from_bytes(hashlib.sha256(payload).digest(), 'big') % FINGERPRINT_PRIME


def multiset_fingerprint(items, challenge):
    result = 1
    for item in items:
        result = (result * (challenge - encode_relation_tuple(item))) % FINGERPRINT_PRIME
    return result


def verify_relation(state, epoch):
    expected, actual = relation_tuples(state)
    # Challenges are generated after candidate construction. Exact
    # Counter equality is retained only as an audit value in this
    # transparent Python notebook.
    rng = np.random.default_rng(SEED + 10_000 + int(epoch))
    challenges = [
        int(rng.integers(2, 2**63, dtype=np.uint64)),
        int(rng.integers(2, 2**63, dtype=np.uint64)),
    ]
    fingerprint_ok = all(
        multiset_fingerprint(expected, challenge)
        == multiset_fingerprint(actual, challenge)
        for challenge in challenges
    )
    exact_audit_ok = Counter(expected) == Counter(actual)
    return bool(fingerprint_ok), bool(exact_audit_ok)


In [ ]:
# CELL 11. Candidate transition, controlled faults, and atomic commit
def state_digest(state):
    digest = hashlib.sha256()
    digest.update(str(state.epoch).encode())
    for record in sorted(state.records, key=lambda value: value['label']):
        digest.update(int(record['label']).to_bytes(8, 'big', signed=True))
        digest.update(int(record['locator']).to_bytes(8, 'big'))
        digest.update(np.packbits(record['root']).tobytes())
    for key in sorted(state.index):
        digest.update(str(key).encode())
        for locator in sorted(state.index[key]):
            digest.update(int(locator).to_bytes(8, 'big'))
    return digest.hexdigest()


def apply_fault(candidate, old_state, fault, key, epoch):
    if fault is None:
        return

    nonempty_keys = [key_value for key_value, values in candidate.index.items() if values]
    first_key = sorted(nonempty_keys)[0]

    if fault == 'omission':
        candidate.index[first_key].pop()
    elif fault == 'duplication':
        candidate.index[first_key].append(candidate.index[first_key][0])
    elif fault == 'locator_corruption':
        candidate.index[first_key][0] ^= 1
    elif fault == 'wrong_bucket':
        locator = candidate.index[first_key].pop()
        block, value = first_key
        wrong_key = (block, (value + 1) % (2**INDEX_BLOCK_BITS))
        candidate.index.setdefault(wrong_key, []).append(locator)
    elif fault == 'partial_update':
        # One index shard remains at the previous epoch.
        candidate.index[first_key][0] = old_state.index[first_key][0]
    elif fault == 'wrong_key':
        wrong = hashlib.sha256(key + b'wrong-index-key').digest()
        for bucket_key, locator_list in old_state.index.items():
            candidate.index[bucket_key] = [
                feistel_prp64(value, wrong, epoch) for value in locator_list
            ]
    else:
        raise ValueError(f'Unknown fault: {fault}')


def sle_transition(old_state, new_epoch, fault=None):
    key = distributed_epoch_key(new_epoch)
    candidate = copy.deepcopy(old_state)
    candidate.epoch = int(new_epoch)

    for record in candidate.records:
        record['locator'] = feistel_prp64(
            record['locator'], key, new_epoch
        )
    for bucket_key, locator_list in candidate.index.items():
        candidate.index[bucket_key] = [
            feistel_prp64(locator, key, new_epoch)
            for locator in locator_list
        ]

    apply_fault(candidate, old_state, fault, key, new_epoch)
    relation_ok, exact_audit_ok = verify_relation(candidate, new_epoch)

    if relation_ok:
        committed = candidate
    else:
        committed = copy.deepcopy(old_state)

    return committed, {
        'epoch': int(new_epoch),
        'fault': fault or 'clean',
        'relation_ok': int(relation_ok),
        'exact_audit_ok': int(exact_audit_ok),
        'committed': int(relation_ok),
        'rollback_preserved_old_state': int(
            relation_ok or state_digest(committed) == state_digest(old_state)
        ),
    }


In [ ]:
# CELL 12. Run clean epochs and verify biometric decision preservation
states = [copy.deepcopy(state0)]
epoch_rows = []
reference_predictions = indexed_before.copy()

current = copy.deepcopy(state0)
for epoch in tqdm(range(1, N_EPOCHS + 1), desc='Clean SLE epochs'):
    current, transition = sle_transition(current, epoch)
    predictions, sizes, recall = indexed_predictions(current, probe_roots)
    row = {
        **transition,
        'rank1': float(np.mean(predictions == probe_labels)),
        'decision_agreement_with_epoch0': float(
            np.mean(predictions == reference_predictions)
        ),
        'true_candidate_recall': float(recall.mean()),
        'mean_candidate_count': float(sizes.mean()),
    }
    epoch_rows.append(row)
    states.append(copy.deepcopy(current))

epoch_df = pd.DataFrame(epoch_rows)
display(epoch_df.head())
print('\nMinimum Rank-1:', epoch_df['rank1'].min())
print('Minimum decision agreement:', epoch_df['decision_agreement_with_epoch0'].min())
print('All relations accepted:', bool(epoch_df['relation_ok'].all()))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epoch_df['epoch'], epoch_df['rank1'], marker='o', label='Rank-1')
axes[0].plot(
    epoch_df['epoch'],
    epoch_df['decision_agreement_with_epoch0'],
    marker='s',
    label='Decision agreement',
)
axes[0].set_ylim(0, 1.03)
axes[0].set_xlabel('SLE epoch')
axes[0].set_ylabel('Score')
axes[0].set_title('(a) Biometric semantics across epochs')
axes[0].legend(frameon=False)
axes[0].grid(alpha=0.25)

locator_history = np.asarray([
    [
        record['locator']
        for record in sorted(state.records, key=lambda value: value['label'])
    ]
    for state in states
], dtype=np.uint64)
for identity_index in range(min(8, N_IDENTITIES)):
    normalized = locator_history[:, identity_index].astype(np.float64) / float(2**64 - 1)
    axes[1].plot(range(N_EPOCHS + 1), normalized, marker='.', linewidth=1)
axes[1].set_xlabel('SLE epoch')
axes[1].set_ylabel('Normalized locator value')
axes[1].set_title('(b) Secret locator evolution (illustrative view)')
axes[1].grid(alpha=0.25)
fig.tight_layout()
epoch_plot_path = OUTPUT / '03_epoch_semantics.png'
fig.savefig(epoch_plot_path, dpi=220, bbox_inches='tight')
plt.show()


In [ ]:
# CELL 15. Controlled relation faults
fault_names = [
    None,
    'omission',
    'duplication',
    'locator_corruption',
    'wrong_bucket',
    'partial_update',
    'wrong_key',
]

fault_rows = []
for offset, fault in enumerate(fault_names, start=1):
    old_copy = copy.deepcopy(state0)
    old_digest = state_digest(old_copy)
    committed, result = sle_transition(
        old_copy,
        new_epoch=100 + offset,
        fault=fault,
    )
    expected_accept = fault is None
    result['expected_relation_ok'] = int(expected_accept)
    result['test_passed'] = int(
        result['relation_ok'] == int(expected_accept)
        and (
            expected_accept
            or state_digest(committed) == old_digest
        )
    )
    fault_rows.append(result)

fault_df = pd.DataFrame(fault_rows)
display(fault_df)
print('All fault expectations passed:', bool(fault_df['test_passed'].all()))

fault_plot = fault_df[fault_df['fault'] != 'clean'].copy()
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.barh(
    fault_plot['fault'],
    1 - fault_plot['relation_ok'],
    color='#159d7a',
)
ax.set_xlim(0, 1.05)
ax.set_xlabel('Fault detection indicator')
ax.set_title('SLE relation validation and atomic rollback')
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
fault_plot_path = OUTPUT / '05_fault_detection.png'
fig.savefig(fault_plot_path, dpi=220, bbox_inches='tight')
plt.show()


## Strong held-out linkability attacks

The next cells generate independently shuffled SLE, private Pointer
Join, and static-locator histories. Attacker features include all old,
new, and XOR bits plus numerical similarity features. Logistic
regression and Extra Trees are trained on five complete runs. Evaluation
uses fifteen different runs and eight late transitions per run.


In [ ]:
# CELL 4. SLE primitives and observable histories
MASK32 = (1 << 32) - 1
MASK64 = (1 << 64) - 1


def epoch_key(run_seed, epoch):
    # Combine five reproducible party contributions into one hidden key.
    contributions = []
    for party in range(5):
        message = f'SLE|run={run_seed}|epoch={epoch}|party={party}'.encode()
        contributions.append(hashlib.sha256(message).digest())
    return hashlib.sha256(b''.join(contributions)).digest()


def feistel_prp64(value, key, epoch, rounds=8):
    # Bijective keyed transform used by the transparent simulator.
    left = (int(value) >> 32) & MASK32
    right = int(value) & MASK32
    for round_no in range(rounds):
        payload = (
            int(epoch).to_bytes(8, 'big')
            + int(round_no).to_bytes(2, 'big')
            + int(right).to_bytes(4, 'big')
        )
        f_value = int.from_bytes(
            hmac.new(key, payload, hashlib.sha256).digest()[:4], 'big'
        )
        left, right = right, (left ^ f_value) & MASK32
    return ((left << 32) | right) & MASK64


def unique_uint64(count, rng):
    values = []
    seen = set()
    while len(values) < count:
        value = int(rng.integers(0, 2**64, dtype=np.uint64))
        if value not in seen:
            seen.add(value)
            values.append(value)
    return np.asarray(values, dtype=np.uint64)


def hide_epoch_order(canonical_history, seed):
    # Return unordered token sets plus hidden ground-truth identities.
    rng = np.random.default_rng(seed)
    tokens = []
    identities = []
    n_identities = canonical_history.shape[1]
    for values in canonical_history:
        permutation = rng.permutation(n_identities)
        tokens.append(values[permutation])
        identities.append(permutation.astype(np.int64))
    return {
        'tokens': np.asarray(tokens, dtype=np.uint64),
        'ids': np.asarray(identities, dtype=np.int64),
    }


def make_sle_history(n_identities, n_epochs, run_seed):
    rng = np.random.default_rng(run_seed)
    current = unique_uint64(n_identities, rng)
    history = [current.copy()]
    for epoch in range(1, n_epochs + 1):
        key = epoch_key(run_seed, epoch)
        current = np.asarray(
            [feistel_prp64(value, key, epoch) for value in current],
            dtype=np.uint64,
        )
        history.append(current.copy())
    return hide_epoch_order(np.asarray(history), run_seed + 91_001)


def make_pointer_join_history(n_identities, n_epochs, run_seed):
    rng = np.random.default_rng(run_seed)
    history = [unique_uint64(n_identities, rng) for _ in range(n_epochs + 1)]
    return hide_epoch_order(np.asarray(history), run_seed + 92_001)


def make_static_history(n_identities, n_epochs, run_seed):
    rng = np.random.default_rng(run_seed)
    initial = unique_uint64(n_identities, rng)
    history = np.repeat(initial[None, :], n_epochs + 1, axis=0)
    return hide_epoch_order(history, run_seed + 93_001)


In [ ]:
# CELL 5. Vectorized 204-dimensional pair features
BYTE_POPCOUNT = np.asarray([int(value).bit_count() for value in range(256)], dtype=np.uint8)


def uint64_bits(values):
    values = np.asarray(values, dtype='>u8').reshape(-1)
    return np.unpackbits(values.view(np.uint8).reshape(-1, 8), axis=1)


def popcount_uint64(values):
    values = np.asarray(values, dtype=np.uint64)
    byte_view = values.view(np.uint8).reshape(values.shape + (8,))
    return BYTE_POPCOUNT[byte_view].sum(axis=-1)


def pair_features(left_values, right_values):
    # Create one feature row for every aligned left/right pair.
    left = np.asarray(left_values, dtype=np.uint64).reshape(-1)
    right = np.asarray(right_values, dtype=np.uint64).reshape(-1)
    if len(left) != len(right):
        raise ValueError('left and right arrays must have equal length')

    left_bits = uint64_bits(left)
    right_bits = uint64_bits(right)
    xor_values = np.bitwise_xor(left, right)
    xor_bits = uint64_bits(xor_values)

    xor_count = popcount_uint64(xor_values).astype(np.float32)
    and_count = popcount_uint64(np.bitwise_and(left, right)).astype(np.float32)
    or_count = popcount_uint64(np.bitwise_or(left, right)).astype(np.float32)
    absolute_difference = (
        np.abs(left.astype(np.float64) - right.astype(np.float64)) / float(2**64 - 1)
    )

    scalar_features = [
        (1.0 - xor_count / 64.0)[:, None],
        (and_count / 64.0)[:, None],
        (or_count / 64.0)[:, None],
        absolute_difference[:, None].astype(np.float32),
    ]
    for prefix_bits in (4, 8, 12, 16, 24, 32, 48):
        shift = 64 - prefix_bits
        equal_prefix = ((left >> np.uint64(shift)) == (right >> np.uint64(shift)))
        scalar_features.append(equal_prefix.astype(np.float32)[:, None])

    left_bytes = left.astype('>u8').view(np.uint8).reshape(-1, 8)
    right_bytes = right.astype('>u8').view(np.uint8).reshape(-1, 8)
    equal_bytes = (left_bytes == right_bytes).mean(axis=1, dtype=np.float32)
    scalar_features.append(equal_bytes[:, None])

    return np.concatenate(
        [
            left_bits.astype(np.float32),
            right_bits.astype(np.float32),
            xor_bits.astype(np.float32),
            *scalar_features,
        ],
        axis=1,
    )


FEATURE_COUNT = pair_features(
    np.asarray([1], dtype=np.uint64),
    np.asarray([2], dtype=np.uint64),
).shape[1]
print('Attacker feature count:', FEATURE_COUNT)


In [ ]:
# CELL 6. Balanced training pairs and full transition matrices
def identity_position_map(identity_array):
    return {int(identity): position for position, identity in enumerate(identity_array)}


def sampled_transition_pairs(history, transition, negatives_per_positive, rng):
    old_tokens = history['tokens'][transition]
    new_tokens = history['tokens'][transition + 1]
    old_ids = history['ids'][transition]
    new_ids = history['ids'][transition + 1]
    new_position = identity_position_map(new_ids)

    left_values = []
    right_values = []
    labels = []
    n_identities = len(old_tokens)

    for old_position, identity in enumerate(old_ids):
        positive_position = new_position[int(identity)]
        left_values.append(old_tokens[old_position])
        right_values.append(new_tokens[positive_position])
        labels.append(1)

        negative_positions = rng.choice(
            [position for position in range(n_identities) if position != positive_position],
            size=min(negatives_per_positive, n_identities - 1),
            replace=False,
        )
        for negative_position in negative_positions:
            left_values.append(old_tokens[old_position])
            right_values.append(new_tokens[int(negative_position)])
            labels.append(0)

    return (
        pair_features(left_values, right_values),
        np.asarray(labels, dtype=np.uint8),
    )


def full_transition_pairs(history, transition):
    old_tokens = history['tokens'][transition]
    new_tokens = history['tokens'][transition + 1]
    old_ids = history['ids'][transition]
    new_ids = history['ids'][transition + 1]
    n_identities = len(old_tokens)

    left_grid = np.repeat(old_tokens, n_identities)
    right_grid = np.tile(new_tokens, n_identities)
    true_matrix = old_ids[:, None] == new_ids[None, :]
    features = pair_features(left_grid, right_grid)
    return features, true_matrix.astype(np.uint8)


def hamming_score_matrix(history, transition):
    old_tokens = history['tokens'][transition]
    new_tokens = history['tokens'][transition + 1]
    xor_matrix = np.bitwise_xor(old_tokens[:, None], new_tokens[None, :])
    return 1.0 - popcount_uint64(xor_matrix).astype(np.float32) / 64.0


def model_score_matrix(model, history, transition):
    features, true_matrix = full_transition_pairs(history, transition)
    n_identities = len(history['tokens'][transition])
    scores = model.predict_proba(features)[:, 1].reshape(n_identities, n_identities)
    return scores, true_matrix


def transition_metrics(scores, true_matrix):
    pair_auc = float(roc_auc_score(true_matrix.ravel(), scores.ravel()))
    row_predictions = np.argmax(scores, axis=1)
    row_top1 = float(np.mean(true_matrix[np.arange(len(row_predictions)), row_predictions]))
    rows, columns = linear_sum_assignment(-scores)
    hungarian_accuracy = float(np.mean(true_matrix[rows, columns]))
    return pair_auc, row_top1, hungarian_accuracy, columns


In [ ]:
# CELL 13. Train on five runs and evaluate fifteen held-out runs
histories = {}
for n_identities in SECURITY_N_VALUES:
    for run in range(N_SECURITY_RUNS):
        run_seed = SEED + n_identities * 10_000 + run * 1_000
        histories[('SLE', n_identities, run)] = make_sle_history(
            n_identities, N_EPOCHS, run_seed + 11
        )
        histories[('Secret Pointer Join', n_identities, run)] = make_pointer_join_history(
            n_identities, N_EPOCHS, run_seed + 22
        )
        histories[('Static locator', n_identities, run)] = make_static_history(
            n_identities, N_EPOCHS, run_seed + 33
        )

attack_rows = []
trained_models = {}
assignment_cache = {}

for n_identities in SECURITY_N_VALUES:
    for condition in ('SLE', 'Secret Pointer Join'):
        rng = np.random.default_rng(SEED + n_identities)
        train_x = []
        train_y = []
        for run in TRAIN_RUNS:
            history = histories[(condition, n_identities, run)]
            for transition in TRAIN_EPOCHS:
                features, labels = sampled_transition_pairs(
                    history, transition, NEGATIVES_PER_POSITIVE, rng
                )
                train_x.append(features)
                train_y.append(labels)
        train_x = np.concatenate(train_x)
        train_y = np.concatenate(train_y)

        print(
            f'Training {condition}, N={n_identities}: '
            f'{len(train_y):,} pairs, {FEATURE_COUNT} features'
        )
        models = {
            'Logistic regression': make_pipeline(
                StandardScaler(),
                LogisticRegression(
                    max_iter=700,
                    class_weight='balanced',
                    solver='liblinear',
                    random_state=SEED,
                ),
            ),
            'Extra Trees': ExtraTreesClassifier(
                n_estimators=EXTRA_TREES,
                max_depth=EXTRA_TREES_MAX_DEPTH,
                min_samples_leaf=4,
                class_weight='balanced_subsample',
                n_jobs=-1,
                random_state=SEED + n_identities,
            ),
        }
        for model in models.values():
            model.fit(train_x, train_y)
        for model_name, model in models.items():
            trained_models[(condition, n_identities, model_name)] = model

        test_jobs = [(run, transition) for run in TEST_RUNS for transition in TEST_EPOCHS]
        for run, transition in tqdm(
            test_jobs,
            desc=f'Attacking {condition}, N={n_identities}',
        ):
            history = histories[(condition, n_identities, run)]
            features, true_matrix = full_transition_pairs(history, transition)

            for model_name, model in models.items():
                scores = model.predict_proba(features)[:, 1].reshape(
                    n_identities, n_identities
                )
                auc, row_top1, global_top1, columns = transition_metrics(
                    scores, true_matrix
                )
                attack_rows.append({
                    'condition': condition,
                    'N': n_identities,
                    'run': run,
                    'transition': transition,
                    'attacker': model_name,
                    'pair_auc': auc,
                    'row_top1': row_top1,
                    'hungarian_accuracy': global_top1,
                })
                if model_name == 'Extra Trees':
                    assignment_cache[(condition, n_identities, run, transition)] = columns

            hamming_scores = hamming_score_matrix(history, transition)
            auc, row_top1, global_top1, _ = transition_metrics(
                hamming_scores, true_matrix
            )
            attack_rows.append({
                'condition': condition,
                'N': n_identities,
                'run': run,
                'transition': transition,
                'attacker': 'Hamming baseline',
                'pair_auc': auc,
                'row_top1': row_top1,
                'hungarian_accuracy': global_top1,
            })

    # Static locator positive control.
    for run in TEST_RUNS:
        history = histories[('Static locator', n_identities, run)]
        for transition in TEST_EPOCHS:
            scores = hamming_score_matrix(history, transition)
            ids_left = history['ids'][transition]
            ids_right = history['ids'][transition + 1]
            true_matrix = (ids_left[:, None] == ids_right[None, :]).astype(np.uint8)
            auc, row_top1, global_top1, _ = transition_metrics(scores, true_matrix)
            attack_rows.append({
                'condition': 'Static locator',
                'N': n_identities,
                'run': run,
                'transition': transition,
                'attacker': 'Hamming baseline',
                'pair_auc': auc,
                'row_top1': row_top1,
                'hungarian_accuracy': global_top1,
            })

attack_df = pd.DataFrame(attack_rows)
print('Attack rows:', len(attack_df))
display(attack_df.head())


## Run-cluster bootstrap

Eight transitions from one history are correlated because they share a
run. The confidence interval therefore first averages all transitions
inside each run and then resamples complete run means. The effective
linkability AUC is reported as `0.5 + abs(mean AUC - 0.5)` because an
attacker can invert a systematically reversed score.


In [ ]:
# CELL 14. Cluster bootstrap over fifteen independent held-out runs
def cluster_bootstrap_mean(group, metric, seed):
    run_means = group.groupby('run')[metric].mean().to_numpy(dtype=np.float64)
    rng = np.random.default_rng(seed)
    draws = rng.choice(
        run_means,
        size=(CLUSTER_BOOTSTRAP_SAMPLES, len(run_means)),
        replace=True,
    ).mean(axis=1)
    return (
        float(run_means.mean()),
        float(np.quantile(draws, 0.025)),
        float(np.quantile(draws, 0.975)),
        int(len(run_means)),
    )


summary_rows = []
for group_index, (keys, group) in enumerate(
    attack_df.groupby(['condition', 'N', 'attacker'])
):
    condition, n_identities, attacker = keys
    row = {
        'condition': condition,
        'N': int(n_identities),
        'attacker': attacker,
        'chance_top1': 1.0 / int(n_identities),
    }
    for metric in ('pair_auc', 'row_top1', 'hungarian_accuracy'):
        mean, low, high, clusters = cluster_bootstrap_mean(
            group, metric, SEED + group_index * 100 + len(metric)
        )
        row[f'{metric}_mean'] = mean
        row[f'{metric}_ci_low'] = low
        row[f'{metric}_ci_high'] = high
        row['held_out_run_clusters'] = clusters
    row['effective_pair_auc'] = 0.5 + abs(row['pair_auc_mean'] - 0.5)
    summary_rows.append(row)

attack_summary_df = pd.DataFrame(summary_rows).sort_values(
    ['condition', 'N', 'attacker']
)
display(attack_summary_df)


In [ ]:
# CELL 15. Sequential multi-epoch tracking from cached graph assignments
tracking_rows = []
for n_identities in SECURITY_N_VALUES:
    for condition in ('SLE', 'Secret Pointer Join'):
        for run in TEST_RUNS:
            history = histories[(condition, n_identities, run)]
            start_transition = TEST_EPOCHS[0]
            predicted_labels = history['ids'][start_transition].copy()

            for horizon, transition in enumerate(TEST_EPOCHS, start=1):
                columns = assignment_cache[
                    (condition, n_identities, run, transition)
                ]
                rows = np.arange(n_identities)
                next_labels = np.empty_like(predicted_labels)
                next_labels[columns] = predicted_labels[rows]
                predicted_labels = next_labels
                true_labels = history['ids'][transition + 1]
                tracking_rows.append({
                    'condition': condition,
                    'N': n_identities,
                    'run': run,
                    'horizon': horizon,
                    'tracking_accuracy': float(np.mean(predicted_labels == true_labels)),
                    'chance': 1.0 / n_identities,
                })

tracking_df = pd.DataFrame(tracking_rows)
tracking_summary_rows = []
for group_index, (keys, group) in enumerate(
    tracking_df.groupby(['condition', 'N', 'horizon'])
):
    condition, n_identities, horizon = keys
    values = group.set_index('run')['tracking_accuracy']
    rng = np.random.default_rng(SEED + 80_000 + group_index)
    array = values.to_numpy()
    draws = rng.choice(
        array,
        size=(CLUSTER_BOOTSTRAP_SAMPLES, len(array)),
        replace=True,
    ).mean(axis=1)
    tracking_summary_rows.append({
        'condition': condition,
        'N': int(n_identities),
        'horizon': int(horizon),
        'tracking_accuracy_mean': float(array.mean()),
        'tracking_ci_low': float(np.quantile(draws, 0.025)),
        'tracking_ci_high': float(np.quantile(draws, 0.975)),
        'chance': 1.0 / int(n_identities),
        'held_out_run_clusters': len(array),
    })
tracking_summary_df = pd.DataFrame(tracking_summary_rows)
display(tracking_summary_df.head(12))


## Real-root bucket leakage ablation

This experiment uses all 128 real CelebA gallery roots. It deliberately
reveals a random fraction of persistent block/value entries in two
epochs and applies Jaccard plus Hungarian matching. It quantifies a
deployment requirement: bucket contents and access patterns must stay
hidden inside MPC/ORAM.


In [ ]:
# CELL 16. Vectorized bucket-metadata side-information attack
def roots_to_block_values(roots, block_bits=INDEX_BLOCK_BITS):
    roots = np.asarray(roots, dtype=np.uint8)
    blocks = roots.reshape(len(roots), -1, block_bits)
    weights = (1 << np.arange(block_bits, dtype=np.uint16))[None, None, :]
    return np.sum(blocks.astype(np.uint16) * weights, axis=2)


block_values = roots_to_block_values(gallery_roots)
n_real, n_blocks = block_values.shape
bucket_rows = []

for fraction in BUCKET_LEAK_FRACTIONS:
    for trial in range(BUCKET_LEAK_TRIALS):
        rng = np.random.default_rng(SEED + trial * 1000 + int(fraction * 10_000))
        permutation = rng.permutation(n_real)

        if fraction == 0.0:
            scores = rng.random((n_real, n_real))
        else:
            old_mask = rng.random((n_real, n_blocks)) < fraction
            new_mask_canonical = rng.random((n_real, n_blocks)) < fraction
            new_mask = new_mask_canonical[permutation]
            new_values = block_values[permutation]

            visible_both = old_mask[:, None, :] & new_mask[None, :, :]
            equal_values = block_values[:, None, :] == new_values[None, :, :]
            intersections = np.sum(visible_both & equal_values, axis=2)
            unions = (
                old_mask.sum(axis=1)[:, None]
                + new_mask.sum(axis=1)[None, :]
                - intersections
            )
            scores = np.divide(
                intersections,
                unions,
                out=np.zeros_like(intersections, dtype=np.float32),
                where=unions > 0,
            )

        true_matrix = (
            np.arange(n_real)[:, None] == permutation[None, :]
        ).astype(np.uint8)
        pair_auc, row_top1, global_top1, _ = transition_metrics(
            scores, true_matrix
        )
        bucket_rows.append({
            'visible_fraction': fraction,
            'trial': trial,
            'pair_auc': pair_auc,
            'row_top1': row_top1,
            'hungarian_accuracy': global_top1,
            'chance_top1': 1.0 / n_real,
        })

bucket_df = pd.DataFrame(bucket_rows)
bucket_summary_df = (
    bucket_df.groupby('visible_fraction', as_index=False)
    .agg(
        pair_auc_mean=('pair_auc', 'mean'),
        pair_auc_std=('pair_auc', 'std'),
        hungarian_accuracy_mean=('hungarian_accuracy', 'mean'),
        hungarian_accuracy_std=('hungarian_accuracy', 'std'),
        chance_top1=('chance_top1', 'first'),
    )
)
display(bucket_summary_df)


In [ ]:
# CELL 17. Compact publication figures with cluster confidence intervals
fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.5))
colors = {'SLE': '#159D7A', 'Secret Pointer Join': '#4C9BD4'}
markers = {'SLE': 'o', 'Secret Pointer Join': 's'}

for condition in ('SLE', 'Secret Pointer Join'):
    subset = attack_summary_df[
        (attack_summary_df['condition'] == condition)
        & (attack_summary_df['attacker'] == 'Extra Trees')
    ].sort_values('N')
    axes[0].errorbar(
        subset['N'], subset['pair_auc_mean'],
        yerr=[
            subset['pair_auc_mean'] - subset['pair_auc_ci_low'],
            subset['pair_auc_ci_high'] - subset['pair_auc_mean'],
        ],
        color=colors[condition], marker=markers[condition], capsize=4,
        label=condition,
    )
    axes[1].errorbar(
        subset['N'], subset['hungarian_accuracy_mean'],
        yerr=[
            subset['hungarian_accuracy_mean'] - subset['hungarian_accuracy_ci_low'],
            subset['hungarian_accuracy_ci_high'] - subset['hungarian_accuracy_mean'],
        ],
        color=colors[condition], marker=markers[condition], capsize=4,
        label=condition,
    )

axes[0].axhline(0.5, color='black', linestyle=':', linewidth=1)
axes[0].set_xlabel('Enrolled identities, N')
axes[0].set_ylabel('Pairwise linkability AUC')
axes[0].set_title('(a) Held-out nonlinear attack')
axes[0].grid(alpha=0.22)
axes[0].legend(frameon=False)

axes[1].plot(
    SECURITY_N_VALUES,
    [1 / value for value in SECURITY_N_VALUES],
    color='black', linestyle=':', label='Chance',
)
axes[1].set_xlabel('Enrolled identities, N')
axes[1].set_ylabel('Hungarian linking accuracy')
axes[1].set_title('(b) Global one-to-one linking')
axes[1].grid(alpha=0.22)
axes[1].legend(frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT / '04_strong_linkability_attacks.png', dpi=240, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.4))
axes[0].errorbar(
    100 * bucket_summary_df['visible_fraction'],
    bucket_summary_df['pair_auc_mean'],
    yerr=bucket_summary_df['pair_auc_std'],
    marker='o', capsize=3, color='#B35C44',
)
axes[0].axhline(0.5, color='black', linestyle=':', linewidth=1)
axes[0].set_xlabel('Visible bucket-signature entries (%)')
axes[0].set_ylabel('Pairwise linkability AUC')
axes[0].set_title('(a) Persistent-metadata leakage')
axes[0].grid(alpha=0.22)

axes[1].errorbar(
    100 * bucket_summary_df['visible_fraction'],
    bucket_summary_df['hungarian_accuracy_mean'],
    yerr=bucket_summary_df['hungarian_accuracy_std'],
    marker='s', capsize=3, color='#7A5AA6',
)
axes[1].axhline(1 / n_real, color='black', linestyle=':', linewidth=1)
axes[1].set_xlabel('Visible bucket-signature entries (%)')
axes[1].set_ylabel('Hungarian linking accuracy')
axes[1].set_title('(b) Global linking from metadata')
axes[1].grid(alpha=0.22)
fig.tight_layout()
fig.savefig(OUTPUT / '05_bucket_metadata_ablation.png', dpi=240, bbox_inches='tight')
plt.show()


In [ ]:
# CELL 18. Falsifiable criteria and complete result archive
criteria_rows = [
    {
        'criterion': 'Real 128-identity Rank-1 remains useful',
        'value': rank1,
        'threshold': '>= 0.80',
        'passed': bool(rank1 >= 0.80),
    },
    {
        'criterion': 'Accepted epochs preserve every decision',
        'value': float(epoch_df['decision_agreement_with_epoch0'].min()),
        'threshold': '== 1.0',
        'passed': bool(epoch_df['decision_agreement_with_epoch0'].min() == 1.0),
    },
    {
        'criterion': 'All controlled relation faults are handled correctly',
        'value': float(fault_df['test_passed'].mean()),
        'threshold': '== 1.0',
        'passed': bool(fault_df['test_passed'].all()),
    },
]

for n_identities in SECURITY_N_VALUES:
    sle = attack_summary_df[
        (attack_summary_df['condition'] == 'SLE')
        & (attack_summary_df['N'] == n_identities)
        & (attack_summary_df['attacker'] == 'Extra Trees')
    ].iloc[0]
    join = attack_summary_df[
        (attack_summary_df['condition'] == 'Secret Pointer Join')
        & (attack_summary_df['N'] == n_identities)
        & (attack_summary_df['attacker'] == 'Extra Trees')
    ].iloc[0]
    static = attack_summary_df[
        (attack_summary_df['condition'] == 'Static locator')
        & (attack_summary_df['N'] == n_identities)
    ].iloc[0]

    criteria_rows.extend([
        {
            'criterion': f'Static positive control detected at N={n_identities}',
            'value': float(static['hungarian_accuracy_mean']),
            'threshold': '== 1.0',
            'passed': bool(static['hungarian_accuracy_mean'] == 1.0),
        },
        {
            'criterion': f'SLE effective learned AUC near random at N={n_identities}',
            'value': float(sle['effective_pair_auc']),
            'threshold': '<= 0.55',
            'passed': bool(sle['effective_pair_auc'] <= 0.55),
        },
        {
            'criterion': f'SLE/private-join effective AUC gap at N={n_identities}',
            'value': float(abs(sle['effective_pair_auc'] - join['effective_pair_auc'])),
            'threshold': '<= 0.03',
            'passed': bool(abs(sle['effective_pair_auc'] - join['effective_pair_auc']) <= 0.03),
        },
        {
            'criterion': f'SLE global linking near chance at N={n_identities}',
            'value': float(sle['hungarian_accuracy_mean']),
            'threshold': f'<= {1 / n_identities + 0.03:.4f}',
            'passed': bool(sle['hungarian_accuracy_mean'] <= 1 / n_identities + 0.03),
        },
    ])

criteria_df = pd.DataFrame(criteria_rows)
display(criteria_df)
print('All empirical journal criteria passed:', bool(criteria_df['passed'].all()))
print('Passing these criteria is empirical evidence, not a cryptographic proof.')

biometric_metrics_df.to_csv(OUTPUT / 'biometric_metrics.csv', index=False)
utility_subset_df.to_csv(OUTPUT / 'utility_identity_subsets.csv', index=False)
epoch_df.to_csv(OUTPUT / 'epoch_semantics.csv', index=False)
fault_df.to_csv(OUTPUT / 'fault_tests.csv', index=False)
attack_df.to_csv(OUTPUT / 'strong_attack_raw.csv', index=False)
attack_summary_df.to_csv(OUTPUT / 'strong_attack_cluster_summary.csv', index=False)
tracking_df.to_csv(OUTPUT / 'multi_epoch_tracking_raw.csv', index=False)
tracking_summary_df.to_csv(OUTPUT / 'multi_epoch_tracking_cluster_summary.csv', index=False)
bucket_df.to_csv(OUTPUT / 'bucket_metadata_ablation_raw.csv', index=False)
bucket_summary_df.to_csv(OUTPUT / 'bucket_metadata_ablation_summary.csv', index=False)
criteria_df.to_csv(OUTPUT / 'success_criteria.csv', index=False)

metadata = {
    'seed': SEED,
    'dataset': CELEBA_DATASET,
    'real_identities': N_IDENTITIES,
    'images_per_identity': IMAGES_PER_ID,
    'gallery_images_per_identity': GALLERY_IMAGES_PER_ID,
    'probe_images': int(len(probe_roots)),
    'embedding_bits': EMBEDDING_BITS,
    'security_N_values': SECURITY_N_VALUES,
    'training_runs': len(TRAIN_RUNS),
    'held_out_run_clusters': len(TEST_RUNS),
    'epochs': N_EPOCHS,
    'test_transitions_per_run': len(TEST_EPOCHS),
    'attacker_feature_count': FEATURE_COUNT,
    'cluster_bootstrap_samples': CLUSTER_BOOTSTRAP_SAMPLES,
    'rank1': rank1,
    'all_empirical_criteria_passed': bool(criteria_df['passed'].all()),
    'claim_boundary': (
        'Empirical unlinkability under observable locator tokens. '
        'No security claim after root, key, or persistent metadata compromise.'
    ),
}
(OUTPUT / 'run_metadata.json').write_text(
    json.dumps(metadata, indent=2), encoding='utf-8'
)

readme = chr(10).join([
    'SLE journal-scale validation',
    '',
    'Biometrics: 128 real CelebA identities, FaceNet/VGGFace2 binary roots.',
    'Security: 5 training runs and 15 independent held-out run clusters.',
    'Statistics: run-cluster bootstrap with 3000 resamples.',
    '',
    'Do not describe empirical attack failure as a cryptographic proof.',
    'Persistent bucket metadata must remain hidden inside MPC/ORAM.',
    '',
])
(OUTPUT / 'README_RESULTS.txt').write_text(readme, encoding='utf-8')

archive_path = shutil.make_archive(
    '/content/sle_journal_validation_results', 'zip', OUTPUT
)
print('Saved result archive:', archive_path)
try:
    from google.colab import files
    files.download(archive_path)
except Exception as exc:
    print('Automatic Colab download unavailable:', exc)


## Expected interpretation

The experiment supports the journal claim only if utility remains
stable, all relation faults are rejected, static locators remain an
easily detectable positive control, learned SLE linkability stays close
to chance across 15 held-out runs, and the result remains comparable to
the private Pointer Join target.

The bucket ablation should be reported separately as a threat-model
finding. It demonstrates why hiding persistent index metadata and access
patterns is a required component of the complete SLE deployment.
